# 85장 · FastAPI 앱 재시작과 DB 호환성

앱을 다시 띄워도 같은 DB를 읽을 수 있으려면 데이터 경로와 스키마가 호환되어야 합니다. 예전에 저장한 행을 새 인스턴스에서 확인합니다.

투두 화면이 열리고도 저장 버튼이 실패할 수 있습니다. 새 코드가 due_date와 priority를 쓰는데 DB에는 82장의 열만 남아 있는 상황입니다. 앱이 요구하는 스키마가 DB에 있는지 확인한 뒤 한 항목의 저장·재조회까지 검증합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0, pydantic 2.14.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 기존 DB 스키마와 앱 버전 확인

제공 구성의 서비스 이름은 db, migrate, api입니다. migrate는 설정할 때만 쓰며 API 시작 검사는 필요한 두 마이그레이션이 적용됐는지 읽습니다. 아래는 082·084 스키마 적용 여부를 확인하는 작은 기록 예제입니다. 파일 내용이 맞는지도 확인하려면 원본처럼 버전별 해시를 대조합니다.


### 셀 01 · 기존 DB 스키마 확인

실제 배포에서는 DB의 schema_migrations에서 적용 이력을 읽습니다. 여기서는 준비 기록을 받아 누락된 버전을 찾습니다.


In [1]:
required = {"082_todo_v1", "084_todo_v2"}
applied = {"082_todo_v1"}
print(sorted(required - applied))

['084_todo_v2']


084_todo_v2가 빠졌습니다. 이 상태에서 화면만 배포 완료로 표시하면 마감일을 저장하는 요청이 DB 경계에서 실패합니다. 운영 DB에 실제로 적용됐는지는 이 예시 기록으로 확인할 수 없습니다.


## 새 앱 인스턴스에서 저장값 재조회

수업용 FastAPI 앱 두 개가 같은 임시 SQLite 파일을 엽니다. 첫 앱에서 v2 필드를 가진 항목을 저장한 뒤, 새 앱의 HTTP 경로에서 같은 ID로 조회합니다. 실제 Compose 재배포와 PostgreSQL 대신 프로세스 안의 서비스·파일 경계를 확인합니다.


### 셀 02 · 앱의 DB 경로와 조회 함수

이 축약본의 POST는 수업에서 정한 제목·마감일·우선순위 입력을 받습니다. 상태가 어디에 저장되는지 보세요.

앞의 배포 기록에서는 084가 빠졌으므로 migrate 단계로 돌아가야 합니다. 다음 셀은 필요한 열을 갖춘 새 SQLite 파일을 사용합니다. 여기에서 확인하는 일은 배포 환경의 이관이 아니라 저장한 값을 새 앱에서 다시 읽는 일입니다.


In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import closing
import sqlite3
from typing import Literal
from datetime import date
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

tmp = TemporaryDirectory()
path = Path(tmp.name) / "release.db"
with closing(sqlite3.connect(path)) as db:
    db.execute("CREATE TABLE todos(id INTEGER PRIMARY KEY, title TEXT, completed INTEGER DEFAULT 0, due_date TEXT, priority TEXT)")
    db.commit()

class TodoInput(BaseModel):
    title: str
    due_date: date | None = None
    priority: Literal["low", "normal", "high"] = "normal"

def make_app(path):
    app = FastAPI()
    @app.post("/todos", status_code=201)
    def add(body: TodoInput):
        with closing(sqlite3.connect(path)) as db:
            cur = db.execute("INSERT INTO todos(title,due_date,priority) VALUES(?,?,?)",
                             (body.title, body.due_date.isoformat() if body.due_date else None, body.priority))
            db.commit()
            return {"data": {"id": cur.lastrowid, "title": body.title, "completed": False,
                             "due_date": body.due_date, "priority": body.priority}}
    @app.get("/todos/{id}")
    def read(id: int):
        with closing(sqlite3.connect(path)) as db:
            row = db.execute("SELECT id,title,completed,due_date,priority FROM todos WHERE id=?", (id,)).fetchone()
        if row is None:
            raise HTTPException(404)
        return {"data": {"id":row[0], "title":row[1], "completed":bool(row[2]), "due_date":row[3], "priority":row[4]}}
    return app

API 객체가 바뀌어도 연결할 파일 경로는 같습니다. 원본 배포 예제의 PostgreSQL 접속 설정과 스키마 검사, 전체 CRUD는 별도 구현에 있습니다.


### 셀 03 · 새 앱에서 같은 ID 재조회

수업 앱 두 개를 차례로 만들고 실제 HTTP 응답 data를 비교합니다.


In [3]:
with TestClient(make_app(path)) as original:
    created = original.post("/todos", json={"title": "복습", "due_date": "2027-04-02", "priority": "high"})
    baseline = created.json()["data"]
with TestClient(make_app(path)) as replacement:
    observed = replacement.get(f"/todos/{baseline['id']}").json()["data"]
print(created.status_code, baseline == observed)
print(observed["title"], observed["due_date"], observed["priority"])

201 True
복습 2027-04-02 high


201 True 뒤에 같은 제목·마감일·우선순위가 나옵니다. 새 앱에서 같은 데이터에 접근했다는 증거입니다. 클라우드 주소나 실제 컨테이너 재시작 결과는 여기에 포함되지 않습니다.


### 과제 04 · 롤백 후 호환성 검사

이전 앱으로 되돌려도 보존할 v1 필드 세 개를 observed에서 꺼내세요. v2 DB 열을 제거하면 안 되는 이유도 생각해 보세요.


In [4]:
# id, title, completed를 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print({key: observed[key] for key in (&#x27;id&#x27;, &#x27;title&#x27;, &#x27;completed&#x27;)})</code></pre>
<p>실행 결과</p><pre><code>{&#x27;id&#x27;: 1, &#x27;title&#x27;: &#x27;복습&#x27;, &#x27;completed&#x27;: False}</code></pre>
<p>기존 앱이 읽던 식별자·제목·완료 상태입니다. 이미지 롤백과 데이터 복구를 분리해야 새 열에 쌓인 마감일·우선순위를 잃지 않습니다. 배포 시험에서는 대상 DB의 마이그레이션 적용 기록과 이전 이미지로 같은 ID를 조회한 결과를 각각 남기세요. 셀의 새 DB 재조회만 끝냈다면 그 행의 값까지만 적으세요.</p>
</details>


## 돌아보기

배포 기록에는 검토한 이미지 식별자, 실제 배포 주소, 스키마 이력, 재조회한 ID와 결과를 적습니다. 이후 최종 프로젝트에서는 배포 여부와 서비스 검증 결과를 각각 근거와 함께 인계합니다.
